In [5]:
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

# Use NEWS_CSV and OUTPUT_DIR from the earlier notebook
coverage = Counter()

reader = pd.read_csv(
    NEWS_CSV,
    usecols=["Date", "Stock_symbol"],
    dtype={"Stock_symbol": "string"},
    chunksize=100_000,
    encoding_errors="replace"
)

for chunk in tqdm(reader, desc="Scanning all stock news"):

    chunk["Stock_symbol"] = (
        chunk["Stock_symbol"].str.strip().str.upper()
    )

    dates = pd.to_datetime(
        chunk["Date"],
        errors="coerce",
        utc=True
    )

    chunk["year"] = dates.dt.year

    chunk = chunk[
        chunk["year"].between(2015, 2023)
        & chunk["Stock_symbol"].notna()
    ]

    counts = chunk.groupby(
        ["Stock_symbol", "year"]
    ).size()

    coverage.update(counts.to_dict())

rows = [
    {"ticker": ticker, "year": year, "articles": count}
    for (ticker, year), count in coverage.items()
]

coverage_df = pd.DataFrame(rows)

pivot = coverage_df.pivot_table(
    index="ticker",
    columns="year",
    values="articles",
    fill_value=0,
    aggfunc="sum"
)

for year in range(2015, 2024):
    if year not in pivot.columns:
        pivot[year] = 0

pivot = pivot[sorted(pivot.columns)]

pivot["total_articles"] = pivot.sum(axis=1)
pivot["years_covered"] = (pivot.loc[:, 2015:2023] > 0).sum(axis=1)
pivot["min_articles_2020_2023"] = pivot.loc[:, 2020:2023].min(axis=1)

pivot = pivot.sort_values(
    ["min_articles_2020_2023", "total_articles"],
    ascending=False
)

pivot.to_csv(OUTPUT_DIR / "all_stock_news_coverage.csv")

display(pivot.head(30))


Scanning all stock news: 156it [03:52,  1.49s/it]


year,2015,2016,2017,2018,2019,2020,2021,2022,2023,total_articles,years_covered,min_articles_2020_2023
ticker,,,,,,,,,,,,
DIS,0,0,0,0,1037,2874,1741,1922,2080,9654,5,1741
WMT,0,0,0,116,1340,2034,1486,1883,1827,8686,6,1486
GOOG,0,0,0,47,2141,2081,1472,1878,2311,9930,6,1472
XOM,0,0,0,101,543,1523,1242,1959,1978,7346,6,1242
INTC,0,0,26,2124,2721,1562,1212,1398,2114,11157,7,1212
T,0,749,1474,1319,1109,1241,1219,1167,1185,9463,8,1167
BRK,104,367,416,389,275,1341,1033,2364,2480,8769,9,1033
GS,0,746,972,805,563,1028,1436,1429,1751,8730,8,1028
CVX,0,339,933,838,739,1057,983,1768,2031,8688,8,983


In [6]:
shortlist = pivot[
    (pivot.loc[:, 2015:2023] >= 100).all(axis=1)
].copy()

print("Stocks meeting threshold:", len(shortlist))

display(shortlist.head(30))

Stocks meeting threshold: 216


year,2015,2016,2017,2018,2019,2020,2021,2022,2023,total_articles,years_covered,min_articles_2020_2023
ticker,,,,,,,,,,,,
BRK,104,367,416,389,275,1341,1033,2364,2480,8769,9,1033
MS,639,619,759,713,551,918,889,1018,1226,7332,9,889
AAL,348,455,551,491,636,1646,912,863,1011,6913,9,863
WFC,419,2133,1928,1095,833,1182,857,850,1186,10483,9,850
BABA,1108,1172,1905,1671,1241,1337,1113,838,1161,11546,9,838
MRK,685,865,1189,1132,849,803,884,1078,980,8465,9,803
CRM,216,911,710,821,688,700,750,863,1466,7125,9,700
NKE,441,672,828,673,540,681,730,921,964,6450,9,681
SBUX,452,698,718,958,805,1165,649,904,886,7235,9,649


In [7]:
shortlist = pivot[
    (pivot.loc[:, 2015:2023] >= 100).all(axis=1)
].copy()

# Sort by total articles (highest first)
shortlist = shortlist.sort_values(
    by="total_articles",
    ascending=False
)

print("Stocks meeting threshold:", len(shortlist))

display(shortlist.head(30))

Stocks meeting threshold: 216


year,2015,2016,2017,2018,2019,2020,2021,2022,2023,total_articles,years_covered,min_articles_2020_2023
ticker,,,,,,,,,,,,
BABA,1108,1172,1905,1671,1241,1337,1113,838,1161,11546,9,838
NVDA,180,337,615,526,508,309,711,2368,5637,11191,9,309
WFC,419,2133,1928,1095,833,1182,857,850,1186,10483,9,850
GILD,1270,1120,1067,1636,1374,2237,399,535,498,10136,9,399
BRK,104,367,416,389,275,1341,1033,2364,2480,8769,9,1033
MRK,685,865,1189,1132,849,803,884,1078,980,8465,9,803
QQQ,785,790,1225,1586,630,496,538,1080,1185,8315,9,496
MU,715,612,1373,1706,1018,560,547,734,882,8147,9,547
KO,786,914,883,783,805,831,556,966,1295,7819,9,556


1. Final selected stocks
Ticker	  Company	         Sector	                      News articles (2015–2023)
NVDA	   NVIDIA	          Information Technology	   11,191
WFC	       Wells Fargo	      Financials	               10,483
MRK	       Merck & Co.	      Healthcare	               8,465
OXY	       Occidental         Petroleum	Energy	           6,544
COST	Costco Wholesale	  Consumer Staples             6,031

SPY	    SPDR S&P 500 ETF Trust	Market benchmark	Separate benchmark

In [8]:
from pathlib import Path
import pandas as pd
import zipfile

BASE_DIR = Path(r"C:\Users\LENOVO\OneDrive\Documents\DataScience\DSTT Project")

PRICE_ZIP = BASE_DIR /"full_history.zip"
NEWS_CSV = BASE_DIR /"nasdaq_exteral_data.csv"

OUTPUT_DIR = BASE_DIR / "filtered_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = ["SPY", "WFC", "COST", "NVDA", "MRK", "OXY"]

START = "2015-01-01"
END_EXCLUSIVE = "2024-01-01"

price_frames = []

with zipfile.ZipFile(PRICE_ZIP) as archive:

    members = {
        Path(name).stem.upper(): name
        for name in archive.namelist()
        if name.lower().endswith(".csv")
    }

    for ticker in TICKERS:

        if ticker not in members:
            print(f"WARNING: {ticker} not found in ZIP")
            continue

        with archive.open(members[ticker]) as file:
            df = pd.read_csv(file)

        df.columns = df.columns.str.strip().str.lower()

        df["date"] = pd.to_datetime(
            df["date"], errors="coerce", utc=True
        )

        df = df[
            (df["date"] >= pd.Timestamp(START, tz="UTC")) &
            (df["date"] < pd.Timestamp(END_EXCLUSIVE, tz="UTC"))
        ].copy()

        df["ticker"] = ticker

        df = df.rename(columns={
            "adj close": "adj_close"
        })

        price_frames.append(df)

if not price_frames:
    raise ValueError("No matching ticker files were found.")

prices = pd.concat(price_frames, ignore_index=True)

prices = prices[
    ["date", "ticker", "open", "high", "low",
     "close", "adj_close", "volume"]
]

prices = prices.drop_duplicates(
    subset=["date", "ticker"]
).sort_values(["ticker", "date"])

prices.to_csv(
    OUTPUT_DIR / "filtered_stock_prices.csv",
    index=False
)

print(prices.groupby("ticker").agg(
    records=("date", "size"),
    first_date=("date", "min"),
    last_date=("date", "max")
))

        records                first_date                 last_date
ticker                                                             
COST       2263 2015-01-02 00:00:00+00:00 2023-12-28 00:00:00+00:00
MRK        2263 2015-01-02 00:00:00+00:00 2023-12-28 00:00:00+00:00
NVDA       2263 2015-01-02 00:00:00+00:00 2023-12-28 00:00:00+00:00
OXY        2263 2015-01-02 00:00:00+00:00 2023-12-28 00:00:00+00:00
SPY        2263 2015-01-02 00:00:00+00:00 2023-12-28 00:00:00+00:00
WFC        2263 2015-01-02 00:00:00+00:00 2023-12-28 00:00:00+00:00


In [9]:
from tqdm.auto import tqdm

NEWS_COLUMNS = [
    "Date",
    "Article_title",
    "Stock_symbol",
    "Url",
    "Publisher",
    "Author",
    "Article"
]

news_frames = []

reader = pd.read_csv(
    NEWS_CSV,
    usecols=NEWS_COLUMNS,
    chunksize=50_000,
    dtype={"Stock_symbol": "string"},
    encoding="utf-8",
    encoding_errors="replace",
    low_memory=False
)

for chunk in tqdm(reader, desc="Filtering Nasdaq news"):

    chunk["Stock_symbol"] = (
        chunk["Stock_symbol"]
        .str.strip()
        .str.upper()
    )

    chunk = chunk[
        chunk["Stock_symbol"].isin(TICKERS)
    ].copy()

    if chunk.empty:
        continue

    chunk["Date"] = pd.to_datetime(
        chunk["Date"],
        errors="coerce",
        utc=True
    )

    chunk = chunk[
        (chunk["Date"] >= pd.Timestamp(START, tz="UTC")) &
        (chunk["Date"] < pd.Timestamp(END_EXCLUSIVE, tz="UTC"))
    ]

    if not chunk.empty:
        news_frames.append(chunk)

if not news_frames:
    raise ValueError("No matching news records found.")

news = pd.concat(news_frames, ignore_index=True)

news = news.rename(columns={
    "Date": "date",
    "Stock_symbol": "ticker",
    "Article_title": "title",
    "Article": "article",
    "Url": "url",
    "Publisher": "publisher",
    "Author": "author"
})

news = news.drop_duplicates(
    subset=["ticker", "date", "url", "title"]
).sort_values(["ticker", "date"])

news.to_csv(
    OUTPUT_DIR / "filtered_financial_news.csv",
    index=False
)

print(news.groupby("ticker").agg(
    articles=("date", "size"),
    first_date=("date", "min"),
    last_date=("date", "max")
))

Filtering Nasdaq news: 311it [02:58,  1.74it/s]


        articles                first_date                 last_date
ticker                                                              
COST        6031 2015-01-02 00:00:00+00:00 2023-12-16 21:00:00+00:00
MRK         8465 2015-01-05 00:00:00+00:00 2023-12-16 10:00:00+00:00
NVDA       11191 2015-01-05 00:00:00+00:00 2023-12-16 20:00:00+00:00
OXY         6526 2015-01-05 00:00:00+00:00 2023-12-16 00:00:00+00:00
SPY         7225 2017-08-18 00:00:00+00:00 2023-12-16 23:00:00+00:00
WFC        10483 2015-01-05 00:00:00+00:00 2023-12-15 00:00:00+00:00


In [10]:
prices["year"] = prices["date"].dt.year
news["year"] = news["date"].dt.year

print("STOCK PRICE COVERAGE")
price_coverage = pd.crosstab(
    prices["year"], prices["ticker"]
)
display(price_coverage)

print("FINANCIAL NEWS COVERAGE")
news_coverage = pd.crosstab(
    news["year"], news["ticker"]
)
display(news_coverage)

price_coverage.to_csv(
    OUTPUT_DIR / "price_annual_coverage.csv"
)

news_coverage.to_csv(
    OUTPUT_DIR / "news_annual_coverage.csv"
)

STOCK PRICE COVERAGE


ticker,COST,MRK,NVDA,OXY,SPY,WFC
year,,,,,,
2015,252,252,252,252,252,252
2016,252,252,252,252,252,252
2017,251,251,251,251,251,251
2018,251,251,251,251,251,251
2019,252,252,252,252,252,252
2020,253,253,253,253,253,253
2021,252,252,252,252,252,252
2022,251,251,251,251,251,251
2023,249,249,249,249,249,249


FINANCIAL NEWS COVERAGE


ticker,COST,MRK,NVDA,OXY,SPY,WFC
year,,,,,,
2015,408,685,180,509,0,419
2016,482,865,337,511,0,2133
2017,757,1189,615,374,1156,1928
2018,547,1132,526,717,2764,1095
2019,438,849,508,1203,0,833
2020,637,803,309,1056,186,1182
2021,525,884,711,362,522,857
2022,1079,1078,2368,962,1300,850
2023,1158,980,5637,832,1297,1186


In [12]:
import pandas as pd

prices = pd.read_csv(r"C:\Users\LENOVO\OneDrive\Documents\DataScience\DSTT Project\filtered_data\filtered_stock_prices.csv")
news = pd.read_csv(r"C:\Users\LENOVO\OneDrive\Documents\DataScience\DSTT Project\filtered_data\filtered_financial_news.csv")

for name, df in [("STOCK PRICES", prices), ("FINANCIAL NEWS", news)]:
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("Missing values:\n", df.isna().sum())
    print("Exact duplicates:", df.duplicated().sum())
    print("\nSample:")
    display(df.head(3))


STOCK PRICES
Rows: 13578
Columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'adj_close', 'volume']
Missing values:
 date         0
ticker       0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64
Exact duplicates: 0

Sample:


,date,ticker,open,high,low,close,adj_close,volume
0,2015-01-02 00:00:00+00:00,COST,141.869995,143.149994,140.809998,141.610001,124.577209,1471000
1,2015-01-05 00:00:00+00:00,COST,141.690002,142.160004,139.800003,140.000000,123.160820,1748500
2,2015-01-06 00:00:00+00:00,COST,140.610001,142.940002,139.850006,141.850006,124.788300,3185200



FINANCIAL NEWS
Rows: 49921
Columns: ['date', 'title', 'ticker', 'url', 'publisher', 'author', 'article']
Missing values:
 date             0
title            0
ticker           0
url              0
publisher    40798
author       49921
article       9124
dtype: int64
Exact duplicates: 0

Sample:


,date,title,ticker,url,publisher,author,article
0,2015-01-02 00:00:00+00:00,Signature Select Pursues Financial Services St...,COST,https://www.nasdaq.com/articles/signature-sele...,NaN,NaN,More than a quarter of Signature Select Canadi...
1,2015-01-03 00:00:00+00:00,Did Your Favorite Retailer Make Its Christmas ...,COST,https://www.nasdaq.com/articles/did-your-favor...,NaN,NaN,Carriers UPS and FedEx were prepared this time...
2,2015-01-05 00:00:00+00:00,"One Put, One Call Option To Know About for Cos...",COST,https://www.nasdaq.com/articles/one-put-one-ca...,NaN,NaN,"Consistently, one of the more popular stocks p..."
